In [6]:
from abc import ABC, abstractmethod
import sympy as sp
import numpy as np
from IPython.display import display, Markdown, Math
from sympy.vector import CoordSys3D

class CLN(ABC):
    def __init__(self, mode='E', a=0.01, sigma=1e6, mur=1):
        # 物理パラメータのデフォルト値を保持
        self.mode=mode
        self.mu_0=4e-7*sp.pi
        self.default_a = a
        self.default_sigma = sigma
        self.default_mu = mur*self.mu_0

        # 物理パラメータ・変数の定義
        self.a = sp.Symbol('a', positive=True)         # radial dimension
        self.s = sp.Symbol('s', positive=True)          # ラプラス演算子
        self.sigma = sp.Symbol('sigma', positive=True) # 導電率
        self.mu = sp.Symbol('mu', positive=True)       # 透磁率
        self.Es = sp.Symbol('E_s', real=True)        # 表面電界の振幅

        self.subs_parameters={self.a: self.default_a,
                             self.sigma: self.default_sigma,
                             self.mu: self.default_mu }

        
        # 内部キャッシュ用（理論解のSymPy式を一元管理）
        self._Z_exact_sym = None
        self.f_exact_num = None

    def displayParameters(self):
        display(Markdown(fr"""
### 📊 現在のパラメータ設定
- **モード (`mode`)**: `{self.mode}`
- **次数 ($n_\theta$)**: `{self.n_value}`
- **半径 (`a`)**: ${sp.latex(self.a)}$ ( `{self.default_a}`)
- **導電率  (${sp.latex(self.sigma)}$)**: ( `{self.default_sigma}`)
- **比透磁率 ($\mu_r$)**:   `{self.default_mu / self.mu_0}`)

"""))
        

    #@abstractmethod
    def solveAmpere(self, Jz, Hs):
        pass

    #@abstractmethod
    def solveFaraday(self, Bt, Es):
        pass

    def solveCurlCurl_A(self, Jz, Es, Hs=0):
        mu = self.mu   
        Hy = self.solveAmpere(Jz, Hs)
        Ez = self.solveFaraday(mu * Hy, Es)
        Az = -Ez
        return Hy, Az

    def solveCurlCurl_T(self, By, Es, Hs=0):
        sigma = self.sigma
        Ez = self.solveFaraday(By, Es)
        Hy = self.solveAmpere(sigma * Ez, Hs)
        return Ez, Hy
        
    @abstractmethod
    def Vintegrate(self, integrand):
        pass

    def We(self, E):
        return self.sigma * E**2
    def Wh(self,H):
        return self.mu * H**2

    def buildCLN(self, variable='E_H', nmax=8, by_integ=False):
        if variable=='E_H': return self.buildCLN_E_H(nmax, by_integ)
        elif variable=='A': return self.buildCLN_A(nmax, by_integ)
        elif variable=='T': return self.buildCLN_T(nmax, by_integ)      
  
    """  E-H Formulation  """
    def buildCLN_E_H(self, nmax=8, by_integ=False):
        self.by_integ=by_integ
        sigma, mu, s= self.sigma, self.mu, self.s
        dZ, dZinv= sp.symbols('dZ dZinv')
        Z = sp.Function('Z')(s)
        E={};H={};R={};L={}
        
        if   self.mode=='E': n=0
        elif self.mode=='H': n=-1
        while n<=nmax:
            if n>=0:
                if n==0:
                    B=0; Es=1;Z=1/dZinv
                else:
                    B= mu*H[n-1]/L[n-1]; Es=0
                E[n]=self.solveFaraday(B, Es) 
                if n>1: 
                    E[n] +=E[n-2]  
                R[n] = 1 / self.Vintegrate(sigma * E[n]**2)
                Z = Z.subs(dZinv,1/( dZ+R[n]))
            
            n=n+1
            if n>nmax: break

            if n>=0:
                if n==0:
                    J=0; Hs=1; Z=dZ
                else:
                    J=sigma * E[n-1]*R[n-1]; Hs=0
                H[n]=self.solveAmpere(J, Hs)
                if n>1:
                    H[n] +=H[n-2]
                L[n]=self.Vintegrate( mu * H[n]**2 )  
                Z = Z.subs(dZ, 1/(1/(s*L[n])+dZinv))
            
            n=n+1

        Z=Z.subs({dZ: 0, dZinv: 0})
        self.R=R
        self.L=L
        return R, L, E, H, Z

    """  A Formulation  """
    def buildCLN_A(self, nmax=8, by_integ=False):
        self.by_integ=by_integ
        sigma, mu, s= self.sigma, self.mu, self.s
        dZ, dZinv= sp.symbols('dZ dZinv')
        Z = sp.Function('Z')(s)
        E={};H={};R={};L={}
        A={}
        
        if   self.mode=='E': n=0
        elif self.mode=='H': n=-1
        while n<=nmax:
            if n>=0:
                if n==0:
                    B=0; Es=1;Z=1/dZinv
                    E[n]=self.solveFaraday(B, Es) 
                else:
                    E[n]=-A[n-1]/L[n-1]
                    if n>1:
                        E[n] +=E[n-2]  
                R[n] = 1 / self.Vintegrate( self.We(E[n]) )
                Z = Z.subs(dZinv,1/( dZ+R[n]))
            
            n=n+1
            if n>nmax: break

            if n>=0:
                if n==0:
                    J=0; Hs=1; Es=0; Z=dZ
                else:
                    J=sigma * E[n-1]*R[n-1]; Es=0; Hs=0
                H[n], A[n]=self.solveCurlCurl_A(J, Es, Hs)
                if n>1:
                    A[n] +=A[n-2]
                    H[n] +=H[n-2]
                L[n]=self.Vintegrate( self.Wh(H[n]) ) 
                Z = Z.subs(dZ, 1/(1/(s*L[n])+dZinv))
            
            n=n+1

        Z=Z.subs({dZ: 0, dZinv: 0})
        self.R=R
        self.L=L
        return R, L, A, E, H, Z

    """  T Formulation  """
    def buildCLN_T(self, nmax=8, by_integ=False):
        self.by_integ=by_integ
        sigma, mu, s= self.sigma, self.mu, self.s
        dZ, dZinv= sp.symbols('dZ dZinv')
        Z = sp.Function('Z')(s)
        E={};H={};R={};L={}
        T={}   
        
        if   self.mode=='E': n=0
        elif self.mode=='H': n=-1
        while n<=nmax:
            if n>=0:
                if n==0:
                    B=0; Es=1; Hs=0; Z=1/dZinv
                else:
                    B= mu*H[n-1]/L[n-1]; Es=0; Hs=0
                E[n], T[n]=self.solveCurlCurl_T(B, Es, Hs)
                if n>1: 
                    E[n] +=E[n-2]  
                    T[n] +=T[n-2]
                R[n] = 1 / self.Vintegrate(sigma * E[n]**2)
                Z = Z.subs(dZinv,1/( dZ+R[n]))
            
            n=n+1
            if n>nmax: break

            if n>=0:
                if n==0:
                    J=0; Hs=1; Z=dZ
                    H[n]=self.solveAmpere(J, Hs)
                else:
                    J=sigma * E[n-1]*R[n-1];
                    H[n]=T[n-1]*R[n-1]
                    if n>1: H[n]+=H[n-2]
                L[n]=self.Vintegrate( mu * H[n]**2 )  
                Z = Z.subs(dZ, 1/(1/(s*L[n])+dZinv))
            
            n=n+1

        Z=Z.subs({dZ: 0, dZinv: 0})
        self.R=R
        self.L=L
        return R, L, T, E, H, Z

    def get_exact_Z_symbolic(self):
        if self._Z_exact_sym is None: 
            self._Z_exact_sym=self.formulation(False)
        return self._Z_exact_sym

    def _format_polynomial_latex(self, expr, z_var):
        """与えられた式を z_var についてのべき数昇順に並べ替え、LaTeX文字列を生成する共通メソッド"""
        try:
            poly = sp.Poly(expr, z_var)
            # 次数の昇順（低い方から高い方へ）にソート
            sorted_terms = sorted(poly.terms(), key=lambda x: x[0], reverse=False)
            
            term_strs = []
            for idx, (deg_t, coeff) in enumerate(sorted_terms):
                deg = deg_t[0]
                c_latex = sp.latex(coeff)
                
                if deg == 0:
                    term_s = c_latex
                elif deg == 1:
                    if coeff == 1: term_s = "z"
                    elif coeff == -1: term_s = "-z"
                    else: term_s = f"{c_latex}z"
                else:
                    if coeff == 1: term_s = f"z^{{{deg}}}"
                    elif coeff == -1: term_s = f"-z^{{{deg}}}"
                    else: term_s = f"{c_latex}z^{{{deg}}}"
                
                # 符号の調整（2番目以降の項でマイナスで始まらない場合は "+" を補う）
                if idx > 0 and not term_s.startswith("-"):
                    term_strs.append(f"+ {term_s}")
                else:
                    term_strs.append(term_s)
                    
            return " ".join(term_strs)
        except Exception:
            return sp.latex(expr)
    """
    def displayA(self, A):
        # r = a*z (z = r/a) に置き換え、電界 E_n(z) を昇順で表示する
        z = sp.symbols('zeta', positive=True)
        r, a = sp.symbols('r a', positive=True) # 必要に応じて定義
        display(Markdown("### **電界モード $A_n(z)$  ( $z = r/a$)**"))
        
        for n, val in A.items():
            val_expr = sp.sympify(val)
            val_z = val_expr.subs(r, a * z).expand()
            #final_latex = self._format_polynomial_latex(val_z, z)
            val_combined = sp.together(val_z)
            final_latex = sp.latex(val_combined)
            display(Math(f"A_{{z,{n}}}(z) = {final_latex}"))    
    """
    def displayE(self, E):
        r=self.x
        z = sp.symbols(r'\zeta', positive=True)
        a = self.a
        display(Markdown(r"### **電界モード $E_{n}(z)$  ( $\zeta = r/a$)**"))
        
        for n, val in E.items():
            val_expr = sp.sympify(val)
            val_z = val_expr.subs(r, a * z)
            final_latex=self._latex_polynomial(val_z)    
            display(Math(fr"E_{{{n}}}(\zeta) = {final_latex}"))
            
    def _latex_polynomial(self, val_z):
        val_z = val_z.expand()
        val_rational = sp.nsimplify(val_z)
        val_combined = sp.together(val_rational) 
        final_latex = sp.latex(val_combined)
        return final_latex
            
    def displayH(self, H):
        r=self.x
        z = sp.symbols(r'\zeta', positive=True)
        a = self.a
        display(Markdown(fr"### **磁界モード $H_{{n}}(z)$   ( $\zeta = r/a$)**"))
        
        isArray=False
        for n, val in H.items():
             if isinstance(val, np.ndarray): 
                 isArray=True
                 break
                 
        
        if isArray:
            for i in [0, 1]:
                for n, val in H.items():
                    val_expr = sp.sympify(val[i])
                    val_z = val_expr.subs(r, a * z)
                    final_latex=self._latex_polynomial(val_z)
                    if i == 0:
                        display(Math(fr"H_{{r, {n}}}(\zeta) = {final_latex}"))
                    else:
                        display(Math(fr"H_{{\theta, {n}}}(\zeta) = {final_latex}"))
        else:
            for n, val in H.items():
                val_expr = sp.sympify(val)
                val_z = val_expr.subs(r, a * z)
                final_latex=self._latex_polynomial(val_z)
                display(Math(fr"H_{{{n}}}(\zeta) = {final_latex}"))

    def plot_EH(self, E, H, z_points=200):
        """電界 E_z と磁界 H_\theta の空間分布をプロットするメソッド（z の範囲: -1 から 1）"""
        import numpy as np
        import matplotlib.pyplot as plt

        z_sym = sp.Symbol(r'\zeta', positive=True)
        a_sym = sp.symbols('a', positive=True)
        #var = sp.Symbol(self.x, real=True)
        var=self.x
        
        # z の範囲を 0 から 1 に設定
        z_vals = np.linspace(0, 1, z_points)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # --- 1. 電界 E_z のプロット ---
        for n, val in E.items():
            expr = sp.sympify(val)
            # 変数を a * z に置き換え、a=1 を代入して z の関数にする
            expr_transformed = expr.subs(var, a_sym * z_sym).subs(a_sym, 1)
            f_lambdified = sp.lambdify(z_sym, expr_transformed, 'numpy')
            
            raw_y = f_lambdified(z_vals)
            
            # 形式に関わらず安全に数値配列へ変換
            try:
                y_vals = np.array(raw_y, dtype=float)
            except (TypeError, ValueError):
                if hasattr(raw_y, 'evalf'):
                    val_f = float(raw_y.evalf())
                else:
                    val_f = float(sp.sympify(raw_y).evalf())
                y_vals = np.full_like(z_vals, val_f)
            
            # 形状チェック：スカラーまたは要素数1の場合は z_vals と同じ長さに拡張
            y_vals = np.atleast_1d(y_vals)
            if y_vals.size == 1:
                y_vals = np.full_like(z_vals, y_vals.item())
            elif y_vals.shape != z_vals.shape:
                y_vals = np.resize(y_vals, z_vals.shape)

            ax1.plot(z_vals, y_vals, label=f"$E_{{z,{n}}}(z)$", linewidth=2)

        ax1.set_title(r"Electric Field $E_{n}(z)$", fontsize=14)
        ax1.set_xlabel(fr"Dimensionless Coordinate $\zeta = {self.x}/a$", fontsize=12)
        ax1.set_ylabel(r"$E_{n}$", fontsize=12)
        ax1.set_xlim(0, 1)
        ax1.grid(True)
        ax1.legend()

        isArray=False
        for n, val in H.items():
             if isinstance(val, np.ndarray): 
                 isArray=True
                 break

        # --- 2. 磁界 H_\theta のプロット ---
        for n, val in H.items():
            if isArray: 
                expr = sp.sympify(val[0])
            else:
                expr = sp.sympify(val)    
            
            expr_transformed = expr.subs(var, a_sym * z_sym).subs(a_sym, 1)
            f_lambdified = sp.lambdify(z_sym, expr_transformed, 'numpy')
            
            raw_y = f_lambdified(z_vals)
            
            try:
                y_vals = np.array(raw_y, dtype=float)
            except (TypeError, ValueError):
                if hasattr(raw_y, 'evalf'):
                    val_f = float(raw_y.evalf())
                else:
                    val_f = float(sp.sympify(raw_y).evalf())
                y_vals = np.full_like(z_vals, val_f)
            
            y_vals = np.atleast_1d(y_vals)
            if y_vals.size == 1:
                y_vals = np.full_like(z_vals, y_vals.item())
            elif y_vals.shape != z_vals.shape:
                y_vals = np.resize(y_vals, z_vals.shape)

            ax2.plot(z_vals, y_vals, label=fr"$H_{{{n}}}(z)$", linewidth=2)

        ax2.set_title(fr"Magnetic Field $H_{{n}}(z)$", fontsize=14)
        ax2.set_xlabel(fr"Dimensionless Coordinate $\zeta = {self.x}/a$", fontsize=12)
        ax2.set_ylabel(r"$H_{n}$", fontsize=12)
        ax2.set_xlim(0, 1)
        ax2.grid(True)
        ax2.legend(loc='upper left')

        plt.tight_layout()
        plt.show()          

    def displayZ(self, Z_expr, title=r"インピーダンス $Z_{\text{CLN}}(s)$ "):
        """R_0, L_1 の定義式および導出された素子値を用いてインピーダンス式を置き換え、表示するメソッド"""

        display(Markdown(f"### **{title}**"))
        
        # 記号の定義
        a, sigma, mu, s = sp.symbols('a sigma mu s', positive=True)
        R_0, L_1 = sp.symbols('R_0 L_1', positive=True)
        dZ, dZinv = sp.symbols('dZ dZinv', commutative=True)
        R_c, L_c = sp.symbols('R_c L_c', positive=True)
        
        Z_val = sp.sympify(Z_expr)
        
        # クラス内に保存されている R, L があればそれを利用、なければデフォルト計算
        #r0_expr = self.R[0] if hasattr(self, 'R') and 0 in self.R else 1 / (sp.pi * a**2 * sigma)
        rc_expr =1/sigma/a
        #l1_expr = self.L[1] if hasattr(self, 'L') and 1 in self.L else mu / (8 * sp.pi)
        lc_expr=mu*a
        
        # 1. 適用する置き換えの式（ルール）の定義
        replacements = {
            dZ: 0,
            dZinv: 0,
            rc_expr: R_c,
            lc_expr: L_c,
        }
        kappa={}
        # R[n] を R[0] で割ることで係数を自動計算し、置換ルールを動的に構築する
        if hasattr(self, 'R'):
            for n, r_val in self.R.items():
                #if n > 0:
                coeff = sp.simplify(r_val / rc_expr)
                target_R = coeff * R_c
                replacements[r_val] = target_R
                replacements[1 / r_val] = 1 / target_R
                kappa[n]=coeff
        
        # L[n] を L[1] で割ることで係数を自動計算し、置換ルールを動的に構築する
        if hasattr(self, 'L'):
            for n, l_val in self.L.items():
                #if n >= 1:
                coeff = sp.simplify(l_val / lc_expr)
                target_L = coeff * L_c
                replacements[l_val] = target_L
                replacements[s * l_val] = s * target_L
                replacements[1 / (s * l_val)] = 1 / (s * target_L)
                kappa[n]=1/coeff


        # 1. n の昇順にソートした項のリストを作成
        kappa_terms = [fr"\kappa_{{{n}}} = {sp.latex(k)}" for n, k in sorted(kappa.items())]
        
        # 2. 2つずつ（2列に）分割する
        rows = []
        for i in range(0, len(kappa_terms), 2):
            # ペアを取り出す（奇数個の場合は slice が自動で調整してくれる）
            pair = kappa_terms[i:i+2]
            # 行ごとに '&' で区切る（aligned 環境で列を分けるため）
            rows.append(" & \\quad ".join(pair))
        
        # 3. aligned 環境でまとめて LaTeX 表示
        latex_str = "\\begin{aligned}\n" + " \\\\\n".join(rows) + "\n\\end{aligned}"
        display(Math(latex_str))
            
        # 2. 置き換えの定義式を動的に表示 (fr"" を使って警告を回避)
        display(Markdown("**【適用する置き換えの式】**"))
        latex_rc = sp.latex(rc_expr)
        latex_lc = sp.latex(lc_expr)
        display(Math(fr"R_c = {latex_rc}, \quad L_c = {latex_lc}"))
        
        # 3. 置換の実行と簡約化
        Z_replaced = Z_val.subs(replacements).simplify()

        self.Z_replaced = Z_replaced
        
        display(sp.Eq(sp.Symbol("Z_{\\text{CLN}}(s)"), Z_replaced))
        return

    def calc_exact_Z(self, omega): 
        s_val = 1j * complex(omega)
        if self.f_exact_num is None:
            s_sym = self.s
            #a_sym, sigma_sym, mu_sym = self.a, self.sigma, self.mu
            Z_sym = self.get_exact_Z_symbolic()
            Z_sub=Z_sym.subs(self.subs_parameters)
            self.f_exact_num = sp.lambdify((s_sym), Z_sub, modules=[self.namespace, 'numpy'])
        return self.f_exact_num(s_val)
       

In [2]:
def plot_Z(self, Z, freq_list=None):
    import matplotlib.pyplot as plt
    if freq_list is None:
        freq_list = np.logspace(2, 6, 500)
        
    omega_list = 2 * np.pi * freq_list
    
    val_a = self.default_a          #getattr(self, 'default_a', 0.01)
    val_sigma = self.default_sigma  #getattr(self, 'default_sigma', 1e6)
    val_mu = self.default_mu        #getattr(self, 'default_mu', 4 * np.pi * 1e-7)
    
    s_sym = sp.symbols('s', complex=True)
    a_sym, sigma_sym, mu_sym = self.a, self.sigma, self.mu  #sp.symbols('a sigma mu', positive=True)
    
    namespace_cln = {
        'besseli': lambda v, x: sp_special.iv(v, x),
        'sqrt': np.sqrt,
        'Pi': np.pi
    }
    print(self.subs_parameters)
    Z_sub=Z.subs(self.subs_parameters)
    f_cln = sp.lambdify((s_sym), Z_sub, modules=[namespace_cln, 'numpy'])
    #f_cln = sp.lambdify((s_sym, a_sym, sigma_sym, mu_sym), Z, modules=[namespace_cln, 'numpy'])
    
    Z_cln_vals = []
    Z_exact_vals = []

    for omega in omega_list:
        fac=1
        if self.mode=='H': 
            fac=1/self.default_a/(1j * omega)/self.default_mu
        val_c = f_cln(1j * omega )*fac
        Z_cln_vals.append(val_c)
        
        val_e = self.calc_exact_Z(omega)*fac
        Z_exact_vals.append(val_e)
        
    Z_cln_vals = np.array(Z_cln_vals, dtype=complex)
    Z_exact_vals = np.array(Z_exact_vals, dtype=complex)
    
    mag_cln = np.abs(Z_cln_vals)
    mag_exact = np.abs(Z_exact_vals)
    phase_cln = np.angle(Z_cln_vals, deg=True)
    phase_exact = np.angle(Z_exact_vals, deg=True)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # 1. 振幅 (Magnitude)
    axes[0].plot(freq_list, mag_cln, label='CLN Approximation', linestyle='--', color='blue', linewidth=2)
    axes[0].plot(freq_list, mag_exact, label='Analytic Solution', linestyle='-', color='red', alpha=0.7, linewidth=2)
    axes[0].set_xscale('log')
    axes[0].set_yscale('log')
    axes[0].set_xlabel('Frequency [Hz]')
    if self.mode=='E':
        axes[0].set_ylabel(' |Z| [$\\Omega$]')
        axes[0].set_title('Impedance |Z| [$\\Omega$]')
    elif self.mode=='H':
        axes[0].set_ylabel(' |$\\mu_r$|')
        axes[0].set_title('Effective relative permeability |$\\mu_r$|')

    axes[0].grid(True, which="both", ls="--")
    axes[0].legend()
    
    # 2. 位相 (Phase)
    axes[1].plot(freq_list, phase_cln, label='CLN Approximation', linestyle='--', color='blue', linewidth=2)
    axes[1].plot(freq_list, phase_exact, label='Analytic Solution', linestyle='-', color='red', alpha=0.7, linewidth=2)
    axes[1].set_xscale('log')
    axes[1].set_xlabel('Frequency [Hz]')
    axes[1].set_ylabel('Phase [deg]')
    if self.mode=='E':
        axes[1].set_title('arg Z [deg]')
    elif self.mode=='H':
        axes[1].set_title('arg $\\mu_r$ [deg]')
    axes[1].grid(True, which="both", ls="--")
    axes[1].legend()
    
    plt.tight_layout()
    plt.show()

CLN.plot_Z = plot_Z

In [3]:
def remove_divergence(self, sol, rp):
    general_sol = sol
    r=self.x
    
    # 1. 一般解に含まれる 'C' から始まる積分定数をすべて取得する
    c_symbols = [s for s in general_sol.free_symbols if s.name.startswith('C')]
    
    constants_to_zero = []
    #print(c_symbols)
    # 2. 積分定数ごとに、それに掛け合わされている「r の関数（係数部分）」を調べる
    for c in c_symbols:
        # 一般解から定数 c の係数を取り出す（例: C2*r**(-n) なら r**(-n) が取れる）
        basis_func = general_sol.coeff(c)
        #print('basis_func=', basis_func)
        
        try:
            # その係数関数だけを r -> 0 で評価する
            lim = sp.limit(basis_func, r, rp)
           
            # もし係数自体が無限大（発散）になるなら、その定数はおとすべき
            if lim.has(sp.oo) or lim.has(-sp.oo) or lim == sp.zoo:
                constants_to_zero.append(c)
        except Exception:
            pass
            
    #print("原点で発散するため 0 にすべき積分定数:", constants_to_zero)
    
    # 3. 発散する定数だけを 0 に置換する
    subs_dict = {c: 0 for c in constants_to_zero}
    sol_regular = general_sol.subs(subs_dict)

    return sol_regular

CLN.remove_divergence = remove_divergence


In [4]:
    def plot_EH(self, E, H, z_points=200):
        """無次元変数 z = r/a (0 <= z <= 1) を用いて電界と磁界の分布を同一の横軸範囲でグラフ描画する"""
        import numpy as np
        import matplotlib.pyplot as plt
        
        z_var = sp.symbols('z', positive=True)
        z_vals = np.linspace(0, 1, z_points)
        r_sym, a_sym = sp.symbols('r a', positive=True)
        
        # 2つのグラフを並べて（または上下に）描画、あるいは軸範囲を完全に一致させる
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # --- (A) 電界 E_n(z) のプロット (左側) ---
        ax1 = axes[0]
        for n, val in E.items():
            val_expr = sp.sympify(val).subs(r_sym, a_sym * z_var).expand()
            val_expr_cleaned = val_expr.cancel()
            f_num = sp.lambdify(z_var, val_expr_cleaned, 'numpy')
            
            y_vals = f_num(z_vals)
            if np.isscalar(y_vals):
                y_vals = np.full_like(z_vals, y_vals)
                
            ax1.plot(z_vals, y_vals, label=f"$E_{{z,{n}}}(z)$", linewidth=2)

        ax1.set_title(r"Electric Field $E_{z,n}(z)$", fontsize=14)
        ax1.set_xlabel(r"Dimensionless Radius $z = r/a$", fontsize=12)
        ax1.set_ylabel(r"$E_{z,n}(z)$", fontsize=12)
        ax1.set_xlim(0, 1)  # 横軸の範囲を 0 から 1 に固定
        ax1.grid(True, linestyle="--", alpha=0.7)
        ax1.axhline(0, color='black', linewidth=0.8, linestyle='-')
        ax1.legend(loc='best', fontsize=9)

        # --- (B) 規格化磁界 H_theta,n(z) のプロット (右側) ---
        norm_factor = 1 / (2 * sp.pi * a_sym)
        ax2 = axes[1]
        for n, val in H.items():
            val_expr = sp.sympify(val).subs(r_sym, a_sym * z_var).expand()
            val_normalized = (val_expr / norm_factor).expand()
            
            f_num = sp.lambdify(z_var, val_normalized, 'numpy')
            
            y_vals = f_num(z_vals)
            if np.isscalar(y_vals):
                y_vals = np.full_like(z_vals, y_vals)
                
            ax2.plot(z_vals, y_vals, label=r"$H_{\theta,%d}(z) / (1/(2\pi a))$" % n, linewidth=2)

        ax2.set_title(r"Normalized Magnetic Field $H_{\theta,n}(z)$", fontsize=14)
        ax2.set_xlabel(r"Dimensionless Radius $z = r/a$", fontsize=12)
        ax2.set_ylabel(r"Normalized $H_{\theta}$", fontsize=12)
        ax2.legend(loc='best', fontsize=9)

        plt.tight_layout()
        plt.show()

In [5]:
import numpy as np
from scipy.special import iv

def calc_F(z_val):
    # 0近傍での特異点を避けるための極限処理（z -> 0 のとき F(z) -> 1）
    sqrt_z = np.sqrt(z_val)
    numer = sqrt_z * iv(0, sqrt_z)
    denom = 2 * iv(1, sqrt_z)
    result = np.where(np.abs(z_val) < 1e-10, 1.0, numer / denom)
    return result

def calc_Z_exact(freqs, R_dc, mu, sigma, a):
    """
    表皮効果を考慮した円柱導体の厳密な内部インピーダンス解析解
    z_val = j * omega * mu * sigma * a^2 (またはこれに類するパラメータ)
    """
    omegas = 2 * np.pi * freqs
    # 表皮効果のパラメータ（定義やスケーリングに合わせて係数を調整してください）
    # 一般的な表式では z_val = 1j * omegas * mu * sigma * a^2 またはその平方などになります
    # ここでは例として一般的な無次元パラメータを使用します
    z_vals = 1j * omegas * mu * sigma * (a**2) # あるいは係数（例: 4など）が入る場合もあります
    
    # calc_F を用いたインピーダンスの計算
    # （※モデルの定式化に合わせて R_dc や係数を乗算します）
    F_vals = calc_F(z_vals)
    
    # 厳密解のインピーダンス（例: R_dc * F_vals など）
    # 導体の表皮効果インピーダンスの厳密解の形に合わせて調整してください
    Z_exact = R_dc * F_vals # 実際のスケーリングに合わせてスカラー倍等を調整
    return Z_exact